# Atelier ML - Partie 3 : phase d'apprentissage

Objectif : entraîner un algorithme à prédire le **pourcentage de victoires** d'un Pokémon à partir de ses caractéristiques.

On veut prédire une **valeur** à partir d'exemples dont on connaît la réponse : c'est un apprentissage **supervisé** de **régression**.

## Modules utilisés

- **Pandas** : chargement et manipulation des données
- **scikit-learn** : algorithmes d'apprentissage, découpage des données et calcul de la précision

In [ ]:
import pandas as pnd

## Chargement du jeu de données

On charge le fichier `dataset.csv` sauvegardé à la fin de la partie 2 (séparateur : tabulations), puis on supprime les lignes qui ont des valeurs manquantes avec `dropna`.

In [ ]:
#Chargement du dataset avec pour séparateur des tabulations
dataset = pnd.read_csv("data/dataset.csv",delimiter='\t')

#Suppression des lignes ayant des valeurs manquantes
dataset = dataset.dropna(axis=0, how='any')

print(dataset.shape)

`dropna` garde **404 Pokémons sur 800**. Il supprime aussi tous les Pokémons qui n'ont pas de second type (`TYPE_2` vide), en plus des 17 Pokémons qui n'ont jamais gagné. C'est la démarche du cours, on la conserve pour obtenir les mêmes résultats.

## Choix des données d'apprentissage

Colonnes de `dataset.csv` (la colonne 0 est l'ancien index du Dataframe) :

| Index | Feature |
|---|---|
| 5 à 11 | `POINTS_DE_VIE`, `NIVEAU_ATTAQUE`, `NIVEAU_DEFENSE`, `NIVEAU_ATTAQUE_SPECIALE`, `NIVEAU_DEFENSE_SPECIALE`, `VITESSE`, `NOMBRE_GENERATIONS` |
| 17 | `POURCENTAGE_DE_VICTOIRES` |

- **X** : les caractéristiques du Pokémon (colonnes 5 à 11, `iloc[:, 5:12]` car la borne de fin est exclue)
- **y** : ce qu'on veut prédire, le pourcentage de victoires (colonne 17)

Le cours utilise aussi l'index 16 dans un de ses exemples, mais dans notre fichier la colonne 16 est `NBR_VICTOIRES` : on utilise donc 17.

In [ ]:
#X = on prend toutes les données, mais uniquement les features 5 à 11
# POINTS_DE_VIE;NIVEAU_ATTAQUE;NIVEAU_DEFENSE;NIVEAU_ATTAQUE_SPECIALE;NIVEAU_DEFENSE_SPECIALE;VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

#y = on prend uniquement la colonne POURCENTAGE_DE_VICTOIRES (17ème feature)
#les : signifiant "Pour toutes les observations"
y = dataset.iloc[:, 17].values

## Jeu d'apprentissage et jeu de tests

`train_test_split` découpe les données : **80 %** pour l'apprentissage et **20 %** pour les tests (`test_size = 0.2`). Les données de test servent à vérifier que le modèle prédit bien des Pokémons qu'il n'a jamais vus. `random_state = 0` donne toujours le même découpage.

In [ ]:
#Construction du jeu d'entrainement et du jeu de tests
from sklearn.model_selection import train_test_split
X_APPRENTISSAGE, X_VALIDATION, Y_APPRENTISSAGE, Y_VALIDATION = train_test_split(X, y, test_size = 0.2, random_state = 0)

print(X_APPRENTISSAGE.shape, X_VALIDATION.shape)

## Algorithme 1 : régression linéaire

La régression linéaire cherche une relation linéaire (une « droite ») entre les caractéristiques X et le pourcentage de victoires y.

On suit toujours le même schéma :

1. `fit` : l'algorithme apprend sur le jeu d'apprentissage
2. `predict` : il fait des prédictions sur le jeu de tests
3. `r2_score` : on compare les prédictions aux vraies valeurs. Plus le score est proche de 1, plus les prédictions sont précises.

In [ ]:
#---- ALGORITHME 1: REGRESSION LINEAIRE -----
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

#Choix de l'algorithme
algorithme = LinearRegression()

#Apprentissage à l'aide de la fonction fit
#Apprentisage effectué sur le jeu de données d'apprentissage
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)

#Realisation de prédictions sur le jeu de tests (validation)
predictions = algorithme.predict(X_VALIDATION)

#Calcul de la précision de l'apprentissage à l'aide de la
#fonction r2_score en comparant les valeurs prédites
#(predictions) et les valeurs attendues (Y_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- REGRESSION LINEAIRE -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

Précision d'environ **0.90** (90 %), la même que dans le cours.

## Algorithme 2 : arbre de décision

Un arbre de décision pose une suite de questions sur les caractéristiques (ex. « la vitesse est-elle supérieure à 90 ? ») pour arriver à une prédiction. On utilise `DecisionTreeRegressor`, la version **régression** de l'arbre, car on prédit une valeur (le pourcentage de victoires) et non une catégorie.

Même schéma que la régression linéaire : `fit`, `predict`, `r2_score`.

In [ ]:
#---- ALGORITHME 2: ARBRE DE DECISION -----
#Choix de l'algorithme
from sklearn.tree import DecisionTreeRegressor
algorithme = DecisionTreeRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- ARBRES DE DECISIONS -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

Précision d'environ **0.87** (87 %, le cours obtient 0.88), un peu moins bonne que la régression linéaire. Le score peut varier légèrement à chaque exécution, car l'arbre fait certains choix au hasard quand plusieurs questions sont aussi bonnes.

## Algorithme 3 : forêt aléatoire

Une forêt aléatoire construit **beaucoup d'arbres de décision**, chacun sur une partie des données prise au hasard, puis fait la moyenne de leurs prédictions. C'est plus fiable qu'un seul arbre.

In [ ]:
#---- ALGORITHME 3: FORETS ALEATOIRES -----
#Choix de l'algorithme
from sklearn.ensemble import RandomForestRegressor
algorithme = RandomForestRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- FORETS ALEATOIRES -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

Précision d'environ **0.93** (93 %) : c'est le meilleur des trois algorithmes.

| Algorithme | Précision |
|---|---|
| Régression linéaire | 0.90 |
| Arbre de décision | 0.87 |
| **Forêt aléatoire** | **0.93** |

Les algorithmes sont utilisés avec leurs paramètres par défaut, sans optimisation.

## Surapprentissage et sous-apprentissage

- **Surapprentissage** (*overfitting*) : le modèle « apprend par cœur » les données d'apprentissage. Il est très précis sur elles, mais beaucoup moins sur des données qu'il n'a jamais vues.
- **Sous-apprentissage** (*underfitting*) : le modèle ne trouve pas de lien entre les données et prédit mal, même sur les données d'apprentissage.

Pour le vérifier, on compare la précision sur le jeu d'apprentissage avec celle sur le jeu de tests.

In [ ]:
predictions = algorithme.predict(X_VALIDATION)
precision_apprentissage = algorithme.score(X_APPRENTISSAGE,Y_APPRENTISSAGE)
precision = r2_score(Y_VALIDATION, predictions)

print(">> Precision apprentissage = "+str(precision_apprentissage))
print(">> Precision tests = "+str(precision))

La précision sur le jeu d'apprentissage (environ 0.99) est plus haute que sur le jeu de tests (environ 0.93), ce qui est normal : le modèle a déjà vu ces données. L'écart reste faible et la précision sur les tests est très bonne : il n'y a pas de surapprentissage important.

## Sauvegarde du modèle

On garde la forêt aléatoire, le meilleur modèle, et on la sauvegarde avec `joblib` dans `modele/modele_pokemon.mod`. Le notebook `partie3_predictions.ipynb` pourra ainsi l'utiliser sans refaire l'apprentissage.

In [ ]:
import joblib
fichier = 'modele/modele_pokemon.mod'
joblib.dump(algorithme, fichier)